In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import itertools
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap
# Optional: higher numeric accuracy (do this before heavy JAX use)
try:
    from jax import config as jax_config
    jax_config.update("jax_enable_x64", True)
except Exception:
    pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('default')
plt.close('all')
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
mpl.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from exojax.opacity.lpf.lpf import voigt

In [ ]:
Line_profiles = {}

xx = jnp.linspace(-9, 9, 2**13)

sigma_low = 1e-3
sigma_high = 1.0
gamma_low = 1e-3
gamma_high = 1.0

params = jnp.array([
    (sigma_low, gamma_low),
    (sigma_high, gamma_low),
    (sigma_low, gamma_high),
    (sigma_high, gamma_high)
])

Line_profiles["Voigt"] = {}
Line_profiles["Voigt"]['np_Faddeva'] = sjx.optical_depth.line_profile.Voigt_np(xx, params[:, 0], params[:, 1])

voigt_batched = jax.jit(
    jax.vmap(lambda s, g: sjx.optical_depth.line_profile.Voigt_JAX_convolution(xx, s, g, oversample=4), in_axes=(0, 0))
)
Line_profiles["Voigt"]["JAX_convolution"] = voigt_batched(params[:, 0], params[:, 1])

batched_voigt = jax.jit(vmap(lambda b, g: voigt(xx, b, g), in_axes=(0, 0)))
Line_profiles["Voigt"]["exojax"] = batched_voigt(params[:, 0], params[:, 1])

sigma_vec = jnp.array([sigma_low, sigma_high])
gamma_vec = jnp.array([gamma_low, gamma_high])

batched_lorentz = jax.jit(jax.vmap(sjx.optical_depth.line_profile.Lorentzian, in_axes=(None, 0)))
Line_profiles["Lorentzian"] = batched_lorentz(xx, gamma_vec)

batched_gaussian = jax.jit(jax.vmap(sjx.optical_depth.line_profile.Gaussian, in_axes=(None, 0)))
Line_profiles["Gaussian"] = batched_gaussian(xx, sigma_vec)

In [ ]:
P = len(params)
colors = np.array(plt.cm.get_cmap("tab10").colors[:P])

fig, (ax, ax_res) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw=dict(height_ratios=[2, 1]))

for ii in range(len(sigma_vec)):
    ax.plot(xx, Line_profiles["Gaussian"][ii], color='k', linestyle=":", linewidth=3)
for ii in range(len(gamma_vec)):
    ax.plot(xx, Line_profiles["Lorentzian"][ii], color='k', linestyle="--", linewidth=3)

for ii in range(P):
    ax.plot(xx, Line_profiles["Voigt"]['np_Faddeva'][ii], color=colors[ii], linestyle="-", linewidth=1.5)
    ax.plot(xx, Line_profiles["Voigt"]['JAX_convolution'][ii], color=colors[ii], linestyle="--", linewidth=3.)
    ax.plot(xx, Line_profiles["Voigt"]['exojax'][ii], color=colors[ii], linestyle=":", linewidth=3.)

for ii in range(P):
    color = colors[ii]
    ax_res.plot(
        xx, Line_profiles["Voigt"]['JAX_convolution'][ii] / Line_profiles["Voigt"]['np_Faddeva'][ii] - 1,
        color=color, linestyle="--", linewidth=1.5
    )
    ax_res.plot(
        xx, Line_profiles["Voigt"]['exojax'][ii] / Line_profiles["Voigt"]['np_Faddeva'][ii] - 1,
        color=color, linestyle=":", linewidth=1.5
    )

# method_handles = [
#     Line2D([0], [0], color="k", lw=8,  label="Scipy Faddeeva"),
#     Line2D([0], [0], color="k", lw=4,  ls="--", label="JAX convolution"),
#     Line2D([0], [0], color="k", lw=2,  ls=":",  label="JAX TG approx."),
# ]
# leg1 = ax.legend(handles=method_handles, loc="upper right", fontsize=16)
# ax.add_artist(leg1)

param_handles = []
for ii in range(P):
    sigma, gamma = params[ii]
    lbl = rf"$\sigma={sigma:g},\ \gamma={gamma:g}$"
    param_handles.append(Line2D([0], [0], color=colors[ii], lw=6, label=lbl))

leg2 = ax.legend(handles=param_handles, loc="upper left", fontsize=16)
ax.add_artist(leg2)

ax.set_title("Voigt profiles")
ax.set_ylabel(r"$V(x;\ \sigma,\ \gamma)$")
ax.set_yscale("log")
ax.set_ylim(1e-6, 1e3)

ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5)
ax_res.set_xlabel("x")        # only bottom panel gets the x-label
ax_res.set_ylabel("residual")
# ax_res.set_yscale("symlog", linthresh=1e-6)

plt.subplots_adjust(hspace=0.05)
fig.tight_layout()
plt.show()

In [ ]:
from skewerjax.constants.lines import get_line, LINES

# 1) Access by alias
lya = get_line("LyA")
print(f"[LyA] {lya.name}  λ0 = {lya.lambda0_A:.5f} Å  (ion={lya.ion}, species={lya.species})")
print(f"     f_osc = {lya.f_osc:.6f},  A = {lya.A_s1:.3e} s^-1,  mass = {lya.mass:.6f} amu")
# print(f"     λ0 in meters = {lya.lambda0_m:.6e} m")

# 2) Access by canonical ID
civ = get_line("CIV-1548")
print(f"[CIV-1548] {civ.name}  λ0 = {civ.lambda0_A:.2f} Å  (ion={civ.ion})")

# 3) List what’s available (first few)
print("\nAvailable line IDs (sample):")
for i, k in enumerate(sorted(LINES.keys())):
    if i >= 10:
        print(" ...")
        break
    L = LINES[k]
    print(f" - {k:>10s} : {L.species} (ion={L.ion})  λ0={L.lambda0_A:.2f} Å")


In [ ]:
lya = sjx.constants.lines.get_line("LyA")
print(f"Lyman-alpha λ₀ = {lya.lambda0_A} Å, f = {lya.f_osc}")